# 01 — Exploratory Data Analysis

This notebook explores the Wikipedia summarization dataset before training.
We analyze:
- Article length distributions
- Title length distributions
- Text quality metrics
- Token budget analysis for T5 / BART / PEGASUS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer

sns.set_theme(style='whitegrid')
DATA_DIR = Path('../pipeline/data/processed')

## 1. Load Data

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
train_df.head(3)

## 2. Text Length Distribution

In [ ]:
train_df['text_words']  = train_df['text'].str.split().str.len()
train_df['title_words'] = train_df['title'].str.split().str.len()
train_df['text_chars']  = train_df['text'].str.len()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(train_df['text_words'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Article Length (words)')
axes[0].set_xlabel('Word count')

axes[1].hist(train_df['title_words'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Title Length (words)')
axes[1].set_xlabel('Word count')

axes[2].hist(train_df['text_chars'], bins=50, color='seagreen', edgecolor='white')
axes[2].set_title('Article Length (characters)')
axes[2].set_xlabel('Char count')

plt.tight_layout()
plt.savefig('eda_length_distributions.png', dpi=150)
plt.show()

print(train_df[['text_words', 'title_words']].describe().round(1))

## 3. Token Budget Analysis

How many samples exceed the 512-token input limit for each model?

In [ ]:
models_to_check = {
    'PEGASUS': 'google/pegasus-xsum',
    'BART':    'facebook/bart-base',
    'T5':      't5-small',
}

sample = train_df.sample(min(500, len(train_df)), random_state=42)

for model_name, hf_path in models_to_check.items():
    tokenizer = AutoTokenizer.from_pretrained(hf_path)
    token_counts = sample['text'].apply(lambda t: len(tokenizer.encode(t, truncation=False)))
    over_limit   = (token_counts > 512).mean() * 100
    print(f'{model_name:<10} | Median tokens: {token_counts.median():.0f} | Over 512-limit: {over_limit:.1f}%')

## 4. Sample Articles

In [ ]:
for i, row in train_df.sample(3, random_state=1).iterrows():
    print(f"TITLE : {row['title']}")
    print(f"TEXT  : {row['text'][:300]}...")
    print('-' * 70)

## 5. Compression Ratio

How much does the title compress the article?

In [ ]:
train_df['compression_ratio'] = train_df['title_words'] / train_df['text_words']

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train_df['compression_ratio'], bins=40, color='mediumpurple', edgecolor='white')
ax.set_title('Compression Ratio (title words / article words)')
ax.set_xlabel('Ratio')
ax.axvline(train_df['compression_ratio'].median(), color='red', linestyle='--', label='Median')
ax.legend()
plt.tight_layout()
plt.savefig('eda_compression_ratio.png', dpi=150)
plt.show()

print(f"Median compression ratio: {train_df['compression_ratio'].median():.3f}")
print(f"Mean compression ratio:   {train_df['compression_ratio'].mean():.3f}")